In [24]:
import pandas as pd
import numpy as np

df = pd.read_csv("../data/fraud_oracle_with_telematics.csv")

df.head()

,Month,WeekOfMonth,DayOfWeek,Make,AccidentArea,DayOfWeekClaimed,MonthClaimed,WeekOfMonthClaimed,Sex,MaritalStatus,...,avg_speed_kmph,max_speed_kmph,hard_brakes_per_trip,rapid_acceleration_events,trip_duration_minutes,distance_km,night_driving_ratio,urban_driving_ratio,harsh_cornering_events,idle_time_minutes
0,Dec,5,Wednesday,Honda,Urban,Tuesday,Jan,1,Female,Single,...,49.967142,73.579620,3,4,63.301642,52.716702,0.145654,0.774319,1,5.252264
1,Jan,3,Wednesday,Honda,Urban,Monday,Jan,4,Male,Single,...,43.617357,62.034068,5,7,27.244272,19.805386,0.071297,0.960082,2,7.935391
2,Oct,5,Friday,Honda,Urban,Thursday,Nov,2,Male,Married,...,51.476885,78.446508,4,4,13.644782,11.706515,0.649385,0.954290,1,1.546263
3,Jun,2,Saturday,Toyota,Rural,Friday,Jul,1,Male,Married,...,60.230299,76.137923,1,8,24.125514,24.218115,0.042510,0.961001,3,0.542277
4,Jan,5,Monday,Honda,Urban,Tuesday,Feb,2,Female,Single,...,42.658466,63.505354,3,2,50.189848,35.683699,0.433904,0.908334,0,4.501436


In [25]:
##For Inspection
print("Shape:", df.shape)
print("\nColumns:\n", df.columns)
print("\nInfo:")
df.info()

Shape: (15420, 43)

Columns:
 Index(['Month', 'WeekOfMonth', 'DayOfWeek', 'Make', 'AccidentArea',
       'DayOfWeekClaimed', 'MonthClaimed', 'WeekOfMonthClaimed', 'Sex',
       'MaritalStatus', 'Age', 'Fault', 'PolicyType', 'VehicleCategory',
       'VehiclePrice', 'FraudFound_P', 'PolicyNumber', 'RepNumber',
       'Deductible', 'DriverRating', 'Days_Policy_Accident',
       'Days_Policy_Claim', 'PastNumberOfClaims', 'AgeOfVehicle',
       'AgeOfPolicyHolder', 'PoliceReportFiled', 'WitnessPresent', 'AgentType',
       'NumberOfSuppliments', 'AddressChange_Claim', 'NumberOfCars', 'Year',
       'BasePolicy', 'avg_speed_kmph', 'max_speed_kmph',
       'hard_brakes_per_trip', 'rapid_acceleration_events',
       'trip_duration_minutes', 'distance_km', 'night_driving_ratio',
       'urban_driving_ratio', 'harsh_cornering_events', 'idle_time_minutes'],
      dtype='str')

Info:
<class 'pandas.DataFrame'>
RangeIndex: 15420 entries, 0 to 15419
Data columns (total 43 columns):
 #   Column     

In [26]:
# Make column names consistent
df.columns = df.columns.str.strip().str.lower()

# Clean text columns (remove spaces, lowercase)
for col in df.select_dtypes(include=['object', 'string']):
    df[col] = df[col].astype(str).str.strip().str.lower()

In [27]:
##For the missing values

# Fill numeric columns with median
num_cols = df.select_dtypes(include=['int64', 'float64']).columns
df[num_cols] = df[num_cols].apply(lambda x: x.fillna(x.median()))

# Fill categorical columns with mode
cat_cols = df.select_dtypes(include=['object', 'string']).columns
for col in cat_cols:
    df[col] = df[col].fillna(df[col].mode()[0])

In [28]:
# Mapping for range-based columns
mapping = {
    'none': 0,
    '0': 0,
    '1 to 7': 3,
    '8 to 15': 10,
    '15 to 30': 20,
    'more than 30': 40
}

range_cols = ['days_policy_claim', 'days_policy_accident']

for col in range_cols:
    if col in df.columns:
        df[col] = df[col].replace(mapping)
        df[col] = pd.to_numeric(df[col])  # convert to actual numbers

In [29]:
#Handle timestamp (telematics)
if 'timestamp' in df.columns:
    df['timestamp'] = pd.to_datetime(df['timestamp'], errors='coerce')

    if 'driver_id' in df.columns:
        df = df.sort_values(by=['driver_id', 'timestamp'])

In [30]:
def clean_range_columns(df):

    mappings = {
        'days_policy_claim': {
            'none': 0, '1 to 7': 3, '8 to 15': 10,
            '15 to 30': 20, 'more than 30': 40
        },
        'pastnumberofclaims': {
            'none': 0, '1': 1, '2 to 4': 3, 'more than 4': 5
        }
    }

    for col, mapping in mappings.items():
        if col in df.columns:
            df[col] = df[col].replace(mapping)
            df[col] = pd.to_numeric(df[col])

    return df


df = clean_range_columns(df)

In [31]:
####Feature Engineering (CLAIM FEATURES)
# Fast claim indicator
if 'days_policy_claim' in df.columns:
    df['fast_claim'] = (df['days_policy_claim'] < 7).astype(int)

# High claim history
if 'pastnumberofclaims' in df.columns:
    df['high_claim_history'] = (df['pastnumberofclaims'] > 2).astype(int)

In [32]:
###Feature Engineering (TELEMATICS)
# Speeding flag
if 'speed' in df.columns:
    df['speed_flag'] = (df['speed'] > 80).astype(int)

# Harsh acceleration
if 'acceleration' in df.columns:
    df['harsh_acc'] = (df['acceleration'] > 3).astype(int)

# Harsh braking
if 'braking' in df.columns:
    df['harsh_brake'] = (df['braking'] > 3).astype(int)

In [34]:
##Time-based features
if 'timestamp' in df.columns:
    df['hour'] = df['timestamp'].dt.hour
    df['day'] = df['timestamp'].dt.day
    df['month'] = df['timestamp'].dt.month

In [35]:
##For checking
print("Final Shape:", df.shape)
df.head()

Final Shape: (15420, 45)


,month,weekofmonth,dayofweek,make,accidentarea,dayofweekclaimed,monthclaimed,weekofmonthclaimed,sex,maritalstatus,...,hard_brakes_per_trip,rapid_acceleration_events,trip_duration_minutes,distance_km,night_driving_ratio,urban_driving_ratio,harsh_cornering_events,idle_time_minutes,fast_claim,high_claim_history
0,dec,5,wednesday,honda,urban,tuesday,jan,1,female,single,...,3,4,63.301642,52.716702,0.145654,0.774319,1,5.252264,0,0
1,jan,3,wednesday,honda,urban,monday,jan,4,male,single,...,5,7,27.244272,19.805386,0.071297,0.960082,2,7.935391,0,0
2,oct,5,friday,honda,urban,thursday,nov,2,male,married,...,4,4,13.644782,11.706515,0.649385,0.954290,1,1.546263,0,0
3,jun,2,saturday,toyota,rural,friday,jul,1,male,married,...,1,8,24.125514,24.218115,0.042510,0.961001,3,0.542277,0,0
4,jan,5,monday,honda,urban,tuesday,feb,2,female,single,...,3,2,50.189848,35.683699,0.433904,0.908334,0,4.501436,0,0


In [36]:
##Saving the processed data
df.to_csv("../data/processed.csv", index=False)

print("✅ Preprocessing complete. File saved as processed.csv")

✅ Preprocessing complete. File saved as processed.csv
